# Voting regressor
Base regressors against a voting regressor on the Boston housing data, then weights, then one algorithm with
different depths. The interactive curve demo is `app.py` (`python app.py`, then open http://127.0.0.1:8050).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

## 1. The data
Boston housing: 506 districts, 13 inputs, median house price MEDV (thousands of dollars).
`load_boston` was removed from scikit-learn in 1.2; the same table is in data/boston.csv (OpenML dataset 531).

In [ ]:
boston = pd.read_csv("data/boston.csv")
X = boston.drop(columns="MEDV")
y = boston["MEDV"]
print(X.shape, y.shape)

## 2. Each base model, 10-fold cross-validation, R²

In [ ]:
lr = LinearRegression()
dt = DecisionTreeRegressor(random_state=0)
svr = SVR()
estimators = [("lr", lr), ("dt", dt), ("svr", svr)]

for name, model in estimators:
    scores = cross_val_score(model, X, y, scoring="r2", cv=10)
    print(name, np.round(np.mean(scores), 2))

## 3. The voting regressor: the mean of the three predictions

In [ ]:
vr = VotingRegressor(estimators)
scores = cross_val_score(vr, X, y, scoring="r2", cv=10)
print("voting regressor", np.round(np.mean(scores), 2))

## 4. Weights from 1 to 3 for each model (27 combinations)

In [ ]:
rows = []
for i in range(1, 4):
    for j in range(1, 4):
        for k in range(1, 4):
            vr = VotingRegressor(estimators, weights=[i, j, k])
            rows.append((i, j, k, np.mean(cross_val_score(vr, X, y, scoring="r2", cv=10))))
weights = pd.DataFrame(rows, columns=["lr", "dt", "svr", "r2"]).round(3).sort_values("r2", ascending=False)
weights.head(8)

## 5. One algorithm, five depths

In [ ]:
trees = [(f"dt{d}", DecisionTreeRegressor(max_depth=d, random_state=0)) for d in [1, 3, 5, 7, None]]
for name, model in trees:
    print(name, np.round(np.mean(cross_val_score(model, X, y, scoring="r2", cv=10)), 2))
print("voting regressor", np.round(np.mean(cross_val_score(VotingRegressor(trees), X, y, scoring="r2", cv=10)), 2))

## 6. Why the scores above are so low: the folds are not shuffled
The Boston rows are stored in order of town, so plain 10-fold cross-validation tests each model on towns it
never saw. Shuffling the rows before cutting the folds gives the usual, much higher scores.

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)
for name, model in estimators + [("voting", VotingRegressor(estimators)), ("voting, depths", VotingRegressor(trees))]:
    print(name, np.round(np.mean(cross_val_score(model, X, y, scoring="r2", cv=cv)), 2))

## 7. The curve demo (same code as app.py)

In [ ]:
from app import figure, run

results = run(["linear regression", "SVR", "decision tree"])
print([(n, round(r2, 2), round(mae, 2)) for n, _, r2, mae in results])
figure(results)

In [ ]:
# Folds in order vs towns: the corrected data (Gilley and Pace 1996, StatLib) names each row's town
from sklearn.model_selection import KFold, GroupKFold
bc = pd.read_csv("data/boston_corrected.txt", sep="\t", skiprows=8, encoding="latin1")
town = bc["TOWN"].values
print("towns:", len(set(town)), " unbroken runs:", 1 + (town[1:] != town[:-1]).sum())
for name, cv in [("in order", KFold(10)), ("shuffled", KFold(10, shuffle=True, random_state=42))]:
    seen = sum(t in set(town[tr]) for tr, te in cv.split(X) for t in town[te])
    print(name, "- test rows whose town is in training:", seen, "of", len(X))
for name, model in estimators + [("voting", VotingRegressor(estimators))]:
    print(name, "GroupKFold by town:", round(cross_val_score(model, X, y, scoring="r2", cv=GroupKFold(10), groups=town).mean(), 2))